In [10]:
# 1. IMPORT LIBRARY

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from collections import Counter

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [11]:
# 2. PENGATURAN DASAR


DATA_DIR = Path("../dataset")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
N_SPLITS = 5

# Memastikan dataset ditemukan
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Folder tidak ditemukan: {DATA_DIR.resolve()}"
    )

# Membaca nama kelas
class_names = sorted([
    folder.name for folder in DATA_DIR.iterdir()
    if folder.is_dir()
])

print("Jumlah kelas:", len(class_names))
print("Daftar kelas:", class_names)

Jumlah kelas: 5
Daftar kelas: ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']


In [13]:
# 3. MEMBACA GAMBAR DAN LABEL


class_to_index = {
    name: index
    for index, name in enumerate(class_names)
}

image_paths = []
labels = []

for class_name in class_names:

    class_folder = DATA_DIR / class_name

    files = [
        file for file in class_folder.rglob("*")
        if file.suffix.lower() in [
            ".jpg", ".jpeg", ".png"
        ]
    ]

    for file in files:
        image_paths.append(str(file))
        labels.append(class_to_index[class_name])

image_paths = np.array(image_paths)
labels = np.array(labels)

if len(image_paths) == 0:
    raise ValueError("Tidak ada gambar yang ditemukan!")

print("Total gambar:", len(image_paths))
print("\nJumlah gambar setiap kelas:")

for index, name in enumerate(class_names):
    print(name, ":", np.sum(labels == index))

Total gambar: 4311

Jumlah gambar setiap kelas:
daisy : 764
dandelion : 1052
rose : 784
sunflower : 731
tulip : 980


In [20]:
# 4. MEMBAGI DATASET
# 70% TRAIN, 15% VALIDATION, 15% TEST


X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("\nJumlah data:")
print("Training   :", len(X_train))
print("Validation :", len(X_val))
print("Testing    :", len(X_test))


Jumlah data:
Training   : 3017
Validation : 647
Testing    : 647


In [21]:
# 5. MEMBACA, RESIZE, DAN NORMALIZATION


def load_image(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    # Resize gambar
    image = tf.image.resize(image, IMG_SIZE)

    # Mengubah tipe data menjadi float32
    image = tf.cast(image, tf.float32)

    # Normalisasi pixel 0-255 menjadi 0-1
    image = image / 255.0

    return image, label

In [22]:
# 6. DATA AUGMENTATION


def create_augmentation():

    return tf.keras.Sequential([

        tf.keras.layers.RandomFlip("horizontal"),

        tf.keras.layers.RandomRotation(0.1),

        tf.keras.layers.RandomZoom(0.1),

        tf.keras.layers.RandomContrast(0.1)

    ], name="data_augmentation")


augmentation = create_augmentation()

In [23]:
# 7. MEMBUAT PIPELINE DATASET


def create_dataset(paths, labels, training=False):

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:

        dataset = dataset.shuffle(
            buffer_size=len(paths),
            seed=SEED
        )

        # Augmentation hanya untuk training
        dataset = dataset.map(
            lambda image, label: (
                augmentation(image, training=True),
                label
            ),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset

In [28]:
# 8. Cross Validation 5 flods

from sklearn.model_selection import StratifiedKFold
import numpy as np

# siapkan seluruh data
image_paths = np.array(image_paths)
labels = np.array(labels)

#pembagi 5 folds 
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

fold_datasets = []
#pembagian data
for fold, (train_idx, val_idx) in enumerate(
    skf.split(image_paths, labels), start=1
):

    print(f"Fold {fold}")

    train_paths = image_paths[train_idx]
    train_labels = labels[train_idx]

    val_paths = image_paths[val_idx]
    val_labels = labels[val_idx]

    train_ds = create_dataset(
        train_paths,
        train_labels,
        training=True
    )

    val_ds = create_dataset(
        val_paths,
        val_labels,
        training=False
    )

    print("Jumlah training:", len(train_paths))
    print("Jumlah validation:", len(val_paths))

Fold 1
Jumlah training: 3448
Jumlah validation: 863
Fold 2
Jumlah training: 3449
Jumlah validation: 862
Fold 3
Jumlah training: 3449
Jumlah validation: 862
Fold 4
Jumlah training: 3449
Jumlah validation: 862
Fold 5
Jumlah training: 3449
Jumlah validation: 862


In [29]:
# 9. FINAL TEST DATASET


test_ds = create_dataset(
    X_test,
    y_test,
    training=False
)

print("Final test dataset berhasil dibuat!")
print("Jumlah data testing:", len(X_test))

Final test dataset berhasil dibuat!
Jumlah data testing: 647


In [30]:
# 10. CEK HASIL PREPROCESSING


print("Preprocessing selesai!")

for fold_data in fold_datasets:

    print(f"\n========== FOLD {fold_data['fold']} ==========")

    images, batch_labels = next(
        iter(fold_data["train_ds"])
    )

    print("Training images:", images.shape)
    print("Training labels:", batch_labels.shape)

    print(
        "Rentang pixel:",
        tf.reduce_min(images).numpy(),
        "-",
        tf.reduce_max(images).numpy()
    )

# Cek final test
test_images, test_labels = next(iter(test_ds))

print("\nFinal test shape:", test_images.shape)
print("Final test labels:", test_labels.shape)

print("\nSEMUA PIPELINE BERHASIL DIBUAT!")

Preprocessing selesai!

Final test shape: (32, 224, 224, 3)
Final test labels: (32,)

SEMUA PIPELINE BERHASIL DIBUAT!


Final test shape (32, 224, 224, 3)
- 32 = jumlah gambar dalam satu batch.
- 224 = tinggi gambar.
- 224 = lebar gambar.
- 3 = jumlah channel warna RGB.

Jadi, program berhasil mengambil 32 gambar berukuran 224 × 224 dengan 3 channel warna.

Final test labels (32,)

Artinya ada 32 label yang sesuai dengan 32 gambar tadi.